In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# ---------- LOAD ----------
df = pd.read_excel('/content/drive/MyDrive/DSPL/Raw/grfc_afi_database_2016-2025_september_update.xlsx')

# ---------- STEP 1: Drop unwanted columns ----------
drop_cols = [
    'Methodology',
    'Source',
    'Selection criteria',
    'Selection in the GRFC',
    'Data availability',
    'Analysis period',
    ' LOWER BOUND Population in Phase 3 or above #',
    'LOWER BOUND Population in Phase 3 or above %',
    'GRFC edition',
    '% pop analysed of tot country pop',
]
df = df.drop(columns=drop_cols)

# ---------- STEP 2: Fix messy column names ----------
df = df.rename(columns={
    'Pupulation group/\nGeographical area analysed': 'Population group/Geographical area analysed'
})
df.columns = df.columns.str.strip()

# ---------- STEP 3: Convert numeric columns ----------
numeric_cols = [
    'Population in Phase 3 or above #', 'Population in Phase 3 or above %',
    'Total country population', 'Population analysed',
    'Population in Phase 1 #', 'Population  in Phase 1 %',
    'Population in Phase 2 #', 'Population  in Phase 2 %',
    'Population in Phase 3 #', 'Population  in Phase 3 %',
    'Population in Phase 4 #', 'Population in Phase 4 %',
    'Population Phase 5 #',    'Population Phase 5 %',
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ---------- STEP 4: Standardise driver columns ----------
for col in ['Primary driver', 'Secondary driver', 'Tertiary driver']:
    df[col] = df[col].str.strip().str.title()
    df[col] = df[col].replace({
        '0': None,
        'Large Number Of Refugees': 'Large Number of Refugees',
    })

# ---------- STEP 5: Fix Population group selected ----------
df['Population group selected'] = df['Population group selected'].replace('-', None)
df['Population group selected'] = df['Population group selected'].str.strip()

# ---------- STEP 6: Drop rows with missing Region ----------
df = df.dropna(subset=['Region'])

# ---------- STEP 7: Remove empty/no-data rows ----------
before = len(df)
df = df.dropna(subset=['Population in Phase 3 or above #'])
print(f"Dropped {before - len(df)} empty rows.")

# ---------- STEP 8: Convert percentages from ratio to percent ----------
pct_cols = [c for c in df.columns if '%' in c]
for col in pct_cols:
    df[col] = df[col] * 100
print(f"Converted {len(pct_cols)} percentage columns to 0-100 format.")

# ---------- FINAL CHECK ----------
print(f"\nFinal shape: {df.shape}")
print(f"Countries: {df['Countries/territories'].nunique()}")
print(f"Year range: {df['Year of reference'].min()} - {df['Year of reference'].max()}")

# ---------- SAVE ----------
df.to_excel('/content/drive/MyDrive/DSPL/Raw/grfc_cleaned_final.xlsx', index=False)
print("\nSaved as grfc_cleaned_final.xlsx in Google Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dropped 780 empty rows.
Converted 6 percentage columns to 0-100 format.

Final shape: (529, 23)
Countries: 74
Year range: 2016 - 2025

Saved as grfc_cleaned_final.xlsx in Google Drive
